In [6]:
!pip install pymoo

In [12]:
import numpy as np
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.sampling.rnd import IntegerRandomSampling
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.repair.rounding import RoundingRepair
from pymoo.optimize import minimize
from pymoo.mcdm.pseudo_weights import PseudoWeights

class EscalaHospitalarProblem(ElementwiseProblem):
    def __init__(self):
        # x1: Medicos Plantonistas (2 a 20), x2: Enfermeiros (4 a 40)
        # TODO 1: Configure 2 variaveis inteiras, 2 objetivos e 1 restricao de desigualdade
        super().__init__(
            n_var=2,
            n_obj=2,
            n_ieq_constr=1,
            xl=np.array([2, 4]),
            xu=np.array([20, 40]),
            vtype=int
        )

    def _evaluate(self, x, out, *args, **kwargs):
        medicos, enfermeiros = x[0], x[1]

        # Objetivo 1: Custo do Plantao (R$)
        f1_custo = (medicos * 1200.0) + (enfermeiros * 450.0)

        # Objetivo 2: Tempo Medio de Espera na Triagem (minutos)
        f2_espera = (180.0 / medicos) + (240.0 / enfermeiros)

        # TODO 2: Restricao de SLA Hospitalar: O tempo de espera (f2_espera) NAO PODE passar de 25 minutos.
        # Lembre-se do formato G(x) <= 0:
        g1 = f2_espera - 25.0

        out["F"] = [f1_custo, f2_espera]
        out["G"] = [g1]

# TODO 3: Instancie o NSGA2 com operadores inteiros (pop_size=40) e rode por 50 geracoes
algoritmo = NSGA2(
    pop_size=40,
    sampling=IntegerRandomSampling(),
    crossover=SBX(vtype=float, repair=RoundingRepair()),
    mutation=PM(vtype=float, repair=RoundingRepair()),
    eliminate_duplicates=True
)

res6 = minimize(
    EscalaHospitalarProblem(),
    algoritmo,
    ('n_gen', 50),
    seed=42,
    verbose=False
)

# TODO 4: Use PseudoWeights com pesos [0.40, 0.60] (40% Custo, 60% Tempo de Espera)
# para selecionar e imprimir a escala recomendada (quantidade de medicos e enfermeiros).
pesos = np.array([0.40, 0.60])
indice_escolhido = PseudoWeights(pesos).do(res6.F)

melhor_x = res6.X[indice_escolhido]
melhor_f = res6.F[indice_escolhido]

print("Escala Hospitalar Recomendada (Pesos 40% Custo / 60% Tempo de Espera):")
print(f"-> Equipe:    {int(melhor_x[0])} Medicos | {int(melhor_x[1])} Enfermeiros")
print(f"-> Resultado: R$ {melhor_f[0]:.2f} | Tempo Medio de Espera: {melhor_f[1]:.2f} min")

Escala Hospitalar Recomendada (Pesos 40% Custo / 60% Tempo de Espera):
-> Equipe:    17 Medicos | 31 Enfermeiros
-> Resultado: R$ 34350.00 | Tempo Medio de Espera: 18.33 min
